# Matched reviewer FDR re-estimation

Run from the package root. Both heavy cells always pass `--resume`; rerunning a cell continues completed method/case/recording units. Preview is the default.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'run_empirical_null_controls.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))

RUN_BH = False
RUN_UNADJUSTED = False
N_SURROGATES = 1000
OUTPUT_ROOT = PACKAGE_ROOT / 'outputs/revision_campaign/empirical_fdr'

COMMON = [
    RUNNER_PYTHON, 'examples/run_empirical_null_controls.py',
    '--cases', 'C,D', '--recordings', 'F3T1,F3T2,F5T2',
    '--representations', 'rise,fall', '--methods', 'cgc,cgc-star',
    '--n-null-replicates', '0', '--n-estimator-surrogates', str(N_SURROGATES),
    '--alpha', '0.05', '--event-mode', 'physical', '--seed', '10', '--resume',
]

def launch(command, enabled):
    print(shlex.join(command))
    if enabled:
        subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)

In [ ]:
bh_command = [*COMMON, '--output-dir', str(OUTPUT_ROOT / 'bh')]
launch(bh_command, RUN_BH)

In [ ]:
unadjusted_command = [*COMMON, '--no-fdr', '--output-dir', str(OUTPUT_ROOT / 'unadjusted')]
launch(unadjusted_command, RUN_UNADJUSTED)

## Verify completed graph artifacts

This lightweight cell checks completion markers and the retained edge-level graph manifest without rerunning any fit.

In [ ]:
import json
import numpy as np

for arm in ('bh', 'unadjusted'):
    arm_dir = OUTPUT_ROOT / arm
    summary_path = arm_dir / 'summary.json'
    manifest_path = arm_dir / 'observed_graph_artifacts_manifest.json'
    if not summary_path.is_file():
        print(f'{arm}: awaiting run ({summary_path})')
        continue
    summary = json.loads(summary_path.read_text())
    if summary.get('status') != 'complete':
        raise RuntimeError(f'{arm}: incomplete summary marker')
    manifest = json.loads(manifest_path.read_text())
    entries = manifest.get('artifacts', [])
    expected = 2 * 2 * 3 * 2  # cases × methods × recordings × representations
    if len(entries) != expected:
        raise RuntimeError(f'{arm}: expected {expected} observed graphs, found {len(entries)}')
    sample = np.load(arm_dir / entries[0]['path'])
    required = {'adjacency', 'retained_scores', 'p_values', 'best_lags', 'metadata_json'}
    if not required.issubset(sample.files):
        raise RuntimeError(f'{arm}: graph artifact is missing required arrays')
    print(f'{arm}: complete; {len(entries)} observed FDR-comparison graph artifacts')